# con_12_route_opt — 실제 도로로 다시 재기 + 정차 추가 가치 판단

1. **실제 도로 거리**로 노선 A(v4)·B+2를 다시 잰다.
2. 그 도로 거리를 써서 **정차를 더 붙일 가치가 있는지** 단계별로 본다.

이미 끝난 것(다시 하지 않음): v4 A·B 확정(report_12), B+2 검증,
B 전면개편·시작점 비교·조합 노선·노선 합치기(전부 기각).

## 도로 데이터를 OSRM으로 바꾼 이유
처음 계획은 osmnx(`network_type="drive"`)였으나 Overpass 공개 서버 3곳이 모두 혼잡해
30분 넘게 구 하나도 받지 못했다. 같은 OSM 데이터를 쓰는 **OSRM 자동차 프로파일**로 바꿨다.
- 원본 데이터는 OSM으로 동일하고 **일방통행을 지킨다**
- OSRM은 **최단 시간** 경로다(osmnx `weight="length"`는 최단 거리).
  버스는 간선도로 위주로 다니므로 이쪽이 노선 현실성에 더 가깝다
- 정류장을 **도로 선 위**에 붙인다(osmnx는 교차점 노드에 붙인다) → 붙인 거리가 더 짧다


In [1]:
import sys, json, time, urllib.parse, urllib.request
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from bus_util import haversine_m
from paths import EXT, PROC

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)

## 0. 노선·후보 정류장 불러오기

In [2]:
# 정류장 좌표
st = pd.read_csv(PROC / "stop_metrics_subway.csv", encoding="utf-8-sig",
                 dtype={"버스정류장ARS번호": str})
C = (st.rename(columns={"버스정류장ARS번호": "ARS"})
     .dropna(subset=["위도", "경도"]).drop_duplicates("ARS").set_index("ARS"))
NM = C["정류장명"].to_dict()

# v4 노선 A·B (report_12) + B+2 (앞에 금호타운·대학동치안센터 추가)
r4 = pd.read_csv(PROC / "fin_route4.csv", encoding="utf-8-sig",
                 dtype={"ARS": str}).sort_values(["축", "순서"])
ROUTE_A = r4[r4["축"] == "A"]["ARS"].tolist()
ROUTE_B = ["21327", "21329"] + r4[r4["축"] == "B"]["ARS"].tolist()
TERM = "23918"

print("A  :", " → ".join(NM[a] for a in ROUTE_A))
print("B+2:", " → ".join(NM[a] for a in ROUTE_B))

A  : 남서울중학교.성보중고등학교 → 난곡입구 → 세이브마트앞 → 우림시장.난곡보건분소 → 난우중학교입구 → 난곡종점 → 난향동주민센터 → 삼성동주민센터 → 봉천사거리 → 방배역.백석예술대학교 → 양재역4번출구
B+2: 금호타운아파트 → 대학동치안센터 → 서울산업정보학교.삼성교 → 관악산입구.관악아트홀·중앙도서관 → 신성초등학교 → 쑥고개입구 → 봉원중학교.행운동우성아파트 → 관악푸르지오아파트 → 방배역.백석예술대학교 → 남부터미널 → 양재역4번출구


In [3]:
# 방향 공백 86곳 (정의: report_08 3단계 그대로)
gap = pd.read_csv(PROC / "gn_gwanak_stops.csv", encoding="utf-8-sig", dtype={"ARS": str})
gap = gap[gap["강남공백"]].copy()
GAP = gap["ARS"].tolist()
GB = dict(zip(gap["ARS"], gap["정류장월승차"]))
GAP_TOT = sum(GB.values())

# 의존 정류장 26곳 (정의: d300 그대로)
fl = pd.read_csv(PROC / "stop_flags_subway.csv", encoding="utf-8-sig",
                 dtype={"버스정류장ARS번호": str}).rename(
                     columns={"버스정류장ARS번호": "ARS"})
dp = fl[(fl["자치구"] == "관악구") & (fl["d300"] == True)]
DEP = set(dp["ARS"])
DB = dict(zip(dp["ARS"], dp["월승차"]))
DEP_TOT = sum(DB.values())

print(f"방향 공백 {len(GAP)}곳 / 승차 {GAP_TOT:,}")
print(f"의존 {len(DEP)}곳 / 승차 {DEP_TOT:,}")

방향 공백 86곳 / 승차 2,663,005
의존 26곳 / 승차 506,393


## 1. 실제 도로로 그리기

OSRM `table` 서비스로 필요한 지점 전체의 **도로 거리 행렬**을 한 번에 받는다.
응답에는 각 지점을 도로에 붙인 거리(`waypoints[].distance`)도 함께 온다.
한 번 받으면 파일로 저장하고 다음부터는 그걸 읽는다.

In [4]:
OSRM = "https://router.project-osrm.org"

def _get(url, tries=4):
    """OSRM 호출 (실패하면 잠깐 쉬고 재시도)."""
    for i in range(tries):
        try:
            with urllib.request.urlopen(url, timeout=120) as r:
                return json.load(r)
        except Exception as e:
            if i == tries - 1:
                raise
            time.sleep(2 * (i + 1))

def osrm_table(pts):
    """좌표 목록의 도로 거리 행렬(m)과 도로에 붙인 거리(m)."""
    s = ";".join(f"{lon:.6f},{lat:.6f}" for lon, lat in pts)
    j = _get(f"{OSRM}/table/v1/driving/{s}?annotations=distance")
    D = np.array(j["distances"], dtype=float)
    snap = [w["distance"] for w in j["sources"]]
    return D, snap

def osrm_path(pts):
    """여러 지점을 순서대로 잇는 도로 경로의 좌표열과 총 거리(m)."""
    s = ";".join(f"{lon:.6f},{lat:.6f}" for lon, lat in pts)
    j = _get(f"{OSRM}/route/v1/driving/{s}?overview=full&geometries=geojson")
    r = j["routes"][0]
    return r["geometry"]["coordinates"], r["distance"]

In [5]:
# 필요한 지점 = 노선 정차 + 공백 86곳
PTS = sorted(set(ROUTE_A + ROUTE_B + GAP))
IDX = {a: i for i, a in enumerate(PTS)}
coords = [(float(C.loc[a, "경도"]), float(C.loc[a, "위도"])) for a in PTS]
print(f"지점 {len(PTS)}곳")

MPATH = EXT / "osrm_dist_matrix.npz"
if MPATH.exists():
    z = np.load(MPATH, allow_pickle=True)
    D, SNAP_A = z["D"], z["snap"]
    assert list(z["pts"]) == PTS, "지점 목록이 바뀌었다 → 파일 지우고 다시 받기"
else:
    D, SNAP_A = osrm_table(coords)
    np.savez(MPATH, D=D, snap=np.array(SNAP_A), pts=np.array(PTS))

SNAP = dict(zip(PTS, SNAP_A))
print(f"행렬 {D.shape} / 못 간 쌍 {int(np.isnan(D).sum())}개")
print(f"도로에 붙인 거리: 최대 {max(SNAP.values()):.0f}m · "
      f"중앙값 {np.median(list(SNAP.values())):.0f}m")

지점 90곳
행렬 (90, 90) / 못 간 쌍 0개
도로에 붙인 거리: 최대 24m · 중앙값 8m


In [6]:
def road(a, b):
    """a→b 도로 거리 m (일방통행 반영)."""
    return float(D[IDX[a], IDX[b]])

def straight(a, b):
    """a→b 직선거리 m (추정)."""
    return float(haversine_m(C.loc[a, "위도"], C.loc[a, "경도"],
                             C.loc[b, "위도"], C.loc[b, "경도"]))

road_len = lambda rt: sum(road(rt[i], rt[i+1]) for i in range(len(rt)-1)) / 1000
straight_len = lambda rt: sum(straight(rt[i], rt[i+1]) for i in range(len(rt)-1)) / 1000

# 노선 구간 중 못 가는 쌍이 있는지 확인
for lab, rt in [("A", ROUTE_A), ("B+2", ROUTE_B)]:
    bad = [(rt[i], rt[i+1]) for i in range(len(rt)-1) if not np.isfinite(road(rt[i], rt[i+1]))]
    print(f"{lab}: 못 가는 구간 {len(bad)}개", bad if bad else "")

A: 못 가는 구간 0개 
B+2: 못 가는 구간 0개 


### 표 1 — 추정(직선) vs 실제 도로

In [7]:
rows = []
for lab, rt in [("A (v4)", ROUTE_A), ("B+2", ROUTE_B)]:
    s_km, r_km = straight_len(rt), road_len(rt)
    end = straight(rt[0], rt[-1]) / 1000          # 굴곡도 분모는 시종점 직선거리
    segs = [(rt[i], rt[i+1], road(rt[i], rt[i+1])) for i in range(len(rt)-1)]
    lo = max(segs, key=lambda x: x[2])
    rows.append({"노선": lab,
                 "지금 값(추정) 길이km": round(s_km, 1),
                 "실제 도로 길이km": round(r_km, 1),
                 "굴곡도(추정)": round(s_km / end, 2),
                 "굴곡도(도로)": round(r_km / end, 2),
                 "가장 긴 구간(도로)km": round(lo[2] / 1000, 1),
                 "가장 긴 구간 위치": f"{NM[lo[0]]}→{NM[lo[1]]}",
                 "정류장-도로 붙인 거리 최대m": round(max(SNAP[a] for a in rt), 1)})
tbl1 = pd.DataFrame(rows)
tbl1

,노선,지금 값(추정) 길이km,실제 도로 길이km,굴곡도(추정),굴곡도(도로),가장 긴 구간(도로)km,가장 긴 구간 위치,정류장-도로 붙인 거리 최대m
0,A (v4),13.9,22.4,1.36,2.18,5.0,봉천사거리→방배역.백석예술대학교,19.2
1,B+2,12.1,17.0,1.35,1.89,6.0,관악푸르지오아파트→방배역.백석예술대학교,20.3


### 표 2 — 구간별 직선 vs 도로 (차이 큰 순 상위 10)

In [8]:
rows = []
for lab, rt in [("A (v4)", ROUTE_A), ("B+2", ROUTE_B)]:
    for i in range(len(rt) - 1):
        a, b = rt[i], rt[i+1]
        s, r = straight(a, b) / 1000, road(a, b) / 1000
        rows.append({"노선": lab, "구간": f"{NM[a]} → {NM[b]}",
                     "직선km": round(s, 2), "도로km": round(r, 2),
                     "차이 배수": round(r / s, 2)})
seg = pd.DataFrame(rows)
tbl2 = seg.sort_values("차이 배수", ascending=False).head(10).reset_index(drop=True)
tbl2

,노선,구간,직선km,도로km,차이 배수
0,A (v4),난곡종점 → 난향동주민센터,0.35,1.03,2.97
1,A (v4),난향동주민센터 → 삼성동주민센터,1.57,3.17,2.03
2,A (v4),난곡입구 → 세이브마트앞,0.92,1.80,1.96
3,A (v4),세이브마트앞 → 우림시장.난곡보건분소,0.42,0.79,1.89
4,A (v4),삼성동주민센터 → 봉천사거리,2.24,4.08,1.82
5,A (v4),남서울중학교.성보중고등학교 → 난곡입구,0.44,0.77,1.75
6,B+2,관악푸르지오아파트 → 방배역.백석예술대학교,3.48,6.04,1.74
7,B+2,신성초등학교 → 쑥고개입구,0.58,0.98,1.69
8,B+2,남부터미널 → 양재역4번출구,1.88,2.95,1.57
9,A (v4),우림시장.난곡보건분소 → 난우중학교입구,0.41,0.62,1.49


In [9]:
print(f"전체 구간 {len(seg)}개 · 차이 배수 중앙값 {seg['차이 배수'].median():.2f} "
      f"· 평균 {seg['차이 배수'].mean():.2f} · 최대 {seg['차이 배수'].max():.2f}")
for _, r in tbl1.iterrows():
    print(f"{r['노선']}: 추정 {r['지금 값(추정) 길이km']}km → 도로 {r['실제 도로 길이km']}km "
          f"({r['실제 도로 길이km']/r['지금 값(추정) 길이km']:.2f}배), "
          f"굴곡도 {r['굴곡도(추정)']} → {r['굴곡도(도로)']}")

전체 구간 20개 · 차이 배수 중앙값 1.48 · 평균 1.55 · 최대 2.97
A (v4): 추정 13.9km → 도로 22.4km (1.61배), 굴곡도 1.36 → 2.18
B+2: 추정 12.1km → 도로 17.0km (1.40배), 굴곡도 1.35 → 1.89


## 2. 정차를 더 붙여볼 가치가 있는지

후보 조건 (전부 만족):
- 관악구 방향 공백 86곳 중 **아직 A·B 어디에도 안 들어간** 곳
- 지금 경로에 **끼워 넣을 수 있는** 곳 (끼워 넣어 늘어나는 **도로** 거리로 계산)
- **되돌아가지 않는** 곳 (종점까지 도로 거리가 앞 정차보다 멀어지지 않음)

매 단계: 남은 후보 중 **늘어난 도로 1km당 늘어나는 방향 공백 승차**가 가장 큰 곳 하나를 고른다.

In [10]:
IN_ROUTE = set(ROUTE_A) | set(ROUTE_B)
CAND0 = [a for a in GAP if a not in IN_ROUTE]
print(f"후보 {len(CAND0)}곳 (공백 86곳 중 이미 정차 {len(GAP)-len(CAND0)}곳 제외)")

dterm = lambda a: road(a, TERM)

def best_insert(route, a):
    """a를 끼워 넣는 가장 싼 자리. (늘어난 m, 넣을 위치) / 불가면 None."""
    best = None
    for i in range(len(route) - 1):
        u, v = route[i], route[i+1]
        d1, d2, d0 = road(u, a), road(a, v), road(u, v)
        if not np.isfinite(d1 + d2 + d0):
            continue
        if dterm(a) > dterm(u):        # 되돌아가면 제외
            continue
        add = d1 + d2 - d0
        if add > 0 and (best is None or add < best[0]):
            best = (add, i + 1)
    return best

def score(rA, rB):
    on = set(rA) | set(rB)
    g = sum(GB[a] for a in GAP if a in on)
    d = sum(DB[a] for a in DEP if a in on)
    return g, g / GAP_TOT * 100, d, d / DEP_TOT * 100

후보 70곳 (공백 86곳 중 이미 정차 16곳 제외)


In [11]:
def greedy(route, other, steps=6):
    """1km당 공백 승차가 가장 큰 후보를 한 곳씩 넣는다."""
    cur, cand = list(route), list(CAND0)
    g, gp, d, dp = score(cur, other)
    out = [{"단계": 0, "추가 정류장": "-", "ARS": "-", "의존?": "-",
            "늘어난 도로 km": 0.0, "방향 공백 % (A+B)": round(gp, 1),
            "의존 %": round(dp, 1), "1km당 늘어난 승차": None,
            "누적 도로 km": round(road_len(cur), 1)}]
    for s in range(1, steps + 1):
        picks = []
        for a in cand:
            bi = best_insert(cur, a)
            if bi is not None:
                picks.append((GB[a] / (bi[0] / 1000), a, bi))
        if not picks:
            print(f"{s}단계: 조건 맞는 후보 없음 → 중단")
            break
        gain, a, (add_m, pos) = max(picks)
        cur = cur[:pos] + [a] + cur[pos:]
        cand.remove(a)
        g, gp, d, dp = score(cur, other)
        out.append({"단계": s, "추가 정류장": NM[a], "ARS": a,
                    "의존?": "O" if a in DEP else "-",
                    "늘어난 도로 km": round(add_m / 1000, 2),
                    "방향 공백 % (A+B)": round(gp, 1), "의존 %": round(dp, 1),
                    "1km당 늘어난 승차": round(gain),
                    "누적 도로 km": round(road_len(cur), 1)})
    return cur, pd.DataFrame(out)

### 표 3 — 노선 B 단계별 (출발점 B+2)

In [12]:
B_final, tbl3 = greedy(ROUTE_B, ROUTE_A, steps=6)
tbl3

,단계,추가 정류장,ARS,의존?,늘어난 도로 km,방향 공백 % (A+B),의존 %,1km당 늘어난 승차,누적 도로 km
0,0,-,-,-,0.00,30.0,8.7,NaN,17.0
1,1,한남운수대학동차고지,21338,-,0.03,30.8,8.7,817358.0,17.0
2,2,관악농협신림지점앞,21180,-,0.21,32.8,8.7,250626.0,17.2
3,3,관악구청,21130,-,0.20,34.1,8.7,173940.0,17.4
4,4,대학동고시촌입구,21157,-,0.31,35.7,8.7,137367.0,17.7
5,5,봉원중학교.행운동우성아파트,21137,-,0.45,37.7,8.7,120048.0,18.2
6,6,영광메디컬센터,21145,-,0.81,41.2,8.7,113545.0,19.0


### 표 4 — 노선 A 단계별 (출발점 A v4)

In [13]:
A_final, tbl4 = greedy(ROUTE_A, ROUTE_B, steps=6)
tbl4

,단계,추가 정류장,ARS,의존?,늘어난 도로 km,방향 공백 % (A+B),의존 %,1km당 늘어난 승차,누적 도로 km
0,0,-,-,-,0.00,30.0,8.7,NaN,22.4
1,1,관악구청,21130,-,0.00,31.3,8.7,355360000.0,22.4
2,2,서울대학교,21128,-,0.02,33.6,8.7,3267120.0,22.4
3,3,관악농협신림지점앞,21180,-,0.03,35.6,8.7,2000229.0,22.4
4,4,미림고개.고시촌입구,21179,-,0.03,36.8,8.7,1044396.0,22.4
5,5,서울대학교,21127,-,0.02,37.4,8.7,958868.0,22.4
6,6,한남운수대학동차고지,21338,-,0.03,38.2,8.7,817358.0,22.5


### 표 5 — 최종 추천안 정차 순서 전체

In [14]:
rows = []
for lab, rt in [("A", A_final), ("B", B_final)]:
    for i, a in enumerate(rt, start=1):
        rows.append({"노선": lab, "순서": i, "ARS": a, "정류장명": NM[a],
                     "월승차": int(C.loc[a, "월승차"]),
                     "의존 여부": "O" if a in DEP else "-",
                     "위도": round(float(C.loc[a, "위도"]), 5),
                     "경도": round(float(C.loc[a, "경도"]), 5)})
tbl5 = pd.DataFrame(rows)
tbl5

,노선,순서,ARS,정류장명,월승차,의존 여부,위도,경도
0,A,1,21116,남서울중학교.성보중고등학교,59125,-,37.48274,126.92084
1,A,2,21115,난곡입구,64668,-,37.48191,126.91594
2,A,3,21207,세이브마트앞,51904,-,37.47386,126.91821
3,A,4,21209,우림시장.난곡보건분소,79988,-,37.47040,126.92002
4,A,5,21210,난우중학교입구,25528,-,37.46669,126.92054
5,A,6,21809,난곡종점,28200,-,37.46413,126.91929
6,A,7,21268,난향동주민센터,24318,-,37.46100,126.91917
7,A,8,21179,미림고개.고시촌입구,33734,-,37.46779,126.93210
8,A,9,21180,관악농협신림지점앞,52406,-,37.47092,126.93380
9,A,10,21165,삼성동주민센터,25789,-,37.46987,126.93295


### 도로 경로 좌표 저장 (PROC/route_road_path.csv)

In [15]:
# 추천안 A·B가 실제로 지나는 도로 좌표를 순서대로 저장
rows = []
for lab, rt in [("A", A_final), ("B", B_final)]:
    pts = [(float(C.loc[a, "경도"]), float(C.loc[a, "위도"])) for a in rt]
    geom, total = osrm_path(pts)
    for i, (lon, lat) in enumerate(geom, start=1):
        rows.append({"노선": lab, "순서": i, "위도": round(lat, 6), "경도": round(lon, 6)})
    print(f"{lab}: 좌표 {len(geom):,}개 / 도로 {total/1000:.1f}km")
rp = pd.DataFrame(rows)
rp.to_csv(PROC / "route_road_path.csv", index=False, encoding="utf-8-sig")
print("저장:", PROC / "route_road_path.csv", rp.shape)

A: 좌표 688개 / 도로 22.5km


B: 좌표 581개 / 도로 18.7km
저장: C:\study\aix\aix_2\data\processed\route_road_path.csv (1269, 4)


### 마지막 네 줄용 수치

In [16]:
def geo(route, label):
    km = road_len(route)
    end = straight(route[0], route[-1]) / 1000
    dts = [dterm(a) for a in route]
    back = [(dts[i+1] - dts[i]) / 1000 for i in range(len(dts)-1) if dts[i+1] > dts[i]]
    lo = max(road(route[i], route[i+1]) for i in range(len(route)-1)) / 1000
    print(f"{label:<10} 도로 {km:5.1f}km  굴곡도 {km/end:4.2f}  가장 긴 구간 {lo:4.1f}km  "
          f"되돌아감 {len(back)}회(최대 {max(back) if back else 0:.2f}km)  정차 {len(route)}곳")

print("[출발점]"); geo(ROUTE_A, "A (v4)"); geo(ROUTE_B, "B+2")
print("\n[추천안]"); geo(A_final, "A 추천"); geo(B_final, "B 추천")

print("\n[커버 — 정차 기준]")
for lab, ra, rb in [("지금 (A + B+2)", ROUTE_A, ROUTE_B), ("추천안", A_final, B_final)]:
    g, gp, d, dp = score(ra, rb)
    print(f"{lab:<16} 방향 공백 {g:>9,} ({gp:4.1f}%)   의존 {d:>8,} ({dp:4.1f}%)")

print("\n[1km당 이득 추이]")
for nmx, t in [("B", tbl3), ("A", tbl4)]:
    print(f"  {nmx}: {[x for x in t['1km당 늘어난 승차'] if pd.notna(x)]}")

[출발점]
A (v4)     도로  22.4km  굴곡도 2.18  가장 긴 구간  5.0km  되돌아감 6회(최대 2.13km)  정차 11곳
B+2        도로  17.0km  굴곡도 1.89  가장 긴 구간  6.0km  되돌아감 2회(최대 1.40km)  정차 11곳

[추천안]
A 추천       도로  22.5km  굴곡도 2.19  가장 긴 구간  5.0km  되돌아감 7회(최대 2.13km)  정차 17곳
B 추천       도로  19.0km  굴곡도 2.11  가장 긴 구간  6.0km  되돌아감 5회(최대 1.59km)  정차 17곳

[커버 — 정차 기준]
지금 (A + B+2)     방향 공백   798,923 (30.0%)   의존   44,271 ( 8.7%)
추천안              방향 공백 1,205,671 (45.3%)   의존   44,271 ( 8.7%)

[1km당 이득 추이]
  B: [817358.0, 250626.0, 173940.0, 137367.0, 120048.0, 113545.0]
  A: [355360000.0, 3267120.0, 2000229.0, 1044396.0, 958868.0, 817358.0]
